# Customer Churn Prediction — End-to-End Project

Upload `customer_churn.csv` before running Cell 2. Run cells top to bottom.

## 1 — Imports

In [ ]:
# Everything used in this project. All of it ships with Google Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42          # one seed, used everywhere, so results are reproducible
print("Imports ready.")

## 2 — Load the data

In [ ]:
# In Colab, upload the file first:
#     from google.colab import files
#     files.upload()          # choose customer_churn.csv
df = pd.read_csv("customer_churn.csv")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])
df.head()

## 3 — Inspect the data

In [ ]:
df.info()
print("\n--- Numeric summary ---")
print(df.describe().round(2))
print("\n--- Target balance ---")
print(df["churn"].value_counts())
print(df["churn"].value_counts(normalize=True).round(3))

## 4 — Data quality

In [ ]:
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

print("\nDuplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Rows after removing duplicates:", len(df))

# customer_id is an identifier, not information. Never let a model use it.
print("\nUnique customer_ids:", df["customer_id"].nunique())

## 5 — EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# 1. Target distribution
df["churn"].value_counts().plot(kind="bar", ax=axes[0, 0], color=["#2E7D57", "#C0392B"])
axes[0, 0].set_title("Churn distribution (0 = stayed, 1 = left)")
axes[0, 0].tick_params(axis="x", rotation=0)

# 2. Churn rate by contract type
df.groupby("contract_type")["churn"].mean().sort_values().plot(
    kind="barh", ax=axes[0, 1], color="#1F4E79")
axes[0, 1].set_title("Churn rate by contract type")

# 3. Churn rate by customer segment
df.groupby("customer_segment")["churn"].mean().plot(
    kind="bar", ax=axes[0, 2], color="#1F4E79")
axes[0, 2].set_title("Churn rate by customer segment")
axes[0, 2].tick_params(axis="x", rotation=0)

# 4. Tenure distribution split by churn
sns.histplot(data=df, x="tenure_months", hue="churn", bins=30,
             ax=axes[1, 0], palette=["#2E7D57", "#C0392B"])
axes[1, 0].set_title("Tenure by churn")

# 5. Satisfaction split by churn
sns.boxplot(data=df, x="churn", y="satisfaction_score", hue="churn",
            legend=False, ax=axes[1, 1], palette=["#2E7D57", "#C0392B"])
axes[1, 1].set_title("Satisfaction score by churn")

# 6. Correlation heatmap (numeric columns only)
num_cols = df.select_dtypes(include=np.number).columns
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=axes[1, 2], annot_kws={"size": 7})
axes[1, 2].set_title("Correlation heatmap")

plt.tight_layout()
plt.show()

# Churn rate by tenure bucket, printed as a table
print(df.groupby(pd.cut(df["tenure_months"], [0, 6, 12, 24, 48, 100]),
                 observed=True)["churn"].agg(["mean", "count"]).round(3))

## 6 — Feature engineering

In [ ]:
# Only transformations with a business justification.
df["tenure_group"] = pd.cut(
    df["tenure_months"], bins=[0, 6, 12, 24, 48, 100],
    labels=["0-6m", "7-12m", "13-24m", "25-48m", "49m+"]).astype(str)

# .clip(lower=1) prevents division by zero producing inf
df["avg_monthly_usage"]   = df["usage_hours"]     / df["tenure_months"].clip(lower=1)
df["support_ticket_rate"] = df["support_tickets"] / df["tenure_months"].clip(lower=1)

print(df[["tenure_months", "tenure_group", "avg_monthly_usage",
          "support_ticket_rate"]].head())

## 7 — Define X and y

In [ ]:
# Drop the target, and drop customer_id (an identifier carries no signal
# and lets the model memorise individuals).
X = df.drop(columns=["churn", "customer_id"])
y = df["churn"]

print("Feature matrix:", X.shape)
print("Churn rate:", round(y.mean(), 3))

## 8 — Train/test split

In [ ]:
# stratify=y keeps the same churn rate in both sets.
# This happens BEFORE any preprocessing — that is what prevents data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print("Train:", X_train.shape, "| churn rate:", round(y_train.mean(), 3))
print("Test :", X_test.shape,  "| churn rate:", round(y_test.mean(), 3))

## 9 — Identify feature types

In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric    :", numeric_features)
print("Categorical:", categorical_features)

## 10 — Numerical pipeline

In [ ]:
# Median imputation (robust to outliers), then scaling.
# Scaling matters for Logistic Regression; it is harmless for trees.
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
print(numeric_transformer)

## 11 — Categorical pipeline

In [ ]:
# Fill gaps with the most common category, then one-hot encode.
# handle_unknown="ignore" stops the pipeline crashing if a category
# appears in production that was never seen during training.
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore")),
])
print(categorical_transformer)

## 12 — ColumnTransformer

In [ ]:
# Routes each group of columns to the right transformer, in one object.
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer,     numeric_features),
    ("cat", categorical_transformer, categorical_features),
])
print(preprocessor)

## 13 — Logistic Regression pipeline

In [ ]:
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
print(logreg_pipeline)

## 14 — Train Logistic Regression

In [ ]:
# .fit() runs the whole chain: impute -> scale -> encode -> train,
# and every step learns from the TRAINING data only.
logreg_pipeline.fit(X_train, y_train)
print("Logistic Regression trained.")

## 15 — Evaluate Logistic Regression

In [ ]:
def evaluate(name, pipeline, X_test, y_test, threshold=0.5):
    """Score a fitted pipeline and return a dict of metrics."""
    proba = pipeline.predict_proba(X_test)[:, 1]
    pred  = (proba >= threshold).astype(int)
    results = {
        "Model":     name,
        "Accuracy":  accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall":    recall_score(y_test, pred),
        "F1":        f1_score(y_test, pred),
        "ROC-AUC":   roc_auc_score(y_test, proba),
    }
    print(f"--- {name} (threshold {threshold}) ---")
    print(classification_report(y_test, pred, target_names=["Stayed", "Churned"]))
    print("Confusion matrix [[TN FP] [FN TP]]:")
    print(confusion_matrix(y_test, pred))
    return results

logreg_results = evaluate("Logistic Regression", logreg_pipeline, X_test, y_test)

## 16 — Decision Tree pipeline

In [ ]:
# max_depth and min_samples_leaf are the anti-overfitting controls.
tree_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=5, min_samples_leaf=20,
                                     random_state=RANDOM_STATE)),
])
print(tree_pipeline)

## 17 — Train Decision Tree

In [ ]:
tree_pipeline.fit(X_train, y_train)
print("Train accuracy:", round(tree_pipeline.score(X_train, y_train), 3))
print("Test  accuracy:", round(tree_pipeline.score(X_test,  y_test),  3))

## 18 — Evaluate Decision Tree

In [ ]:
tree_results = evaluate("Decision Tree", tree_pipeline, X_test, y_test)

## 19 — Random Forest pipeline

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=300, max_depth=10,
                                     min_samples_leaf=5,
                                     random_state=RANDOM_STATE, n_jobs=-1)),
])
print(rf_pipeline)

## 20 — Train Random Forest

In [ ]:
rf_pipeline.fit(X_train, y_train)
print("Train accuracy:", round(rf_pipeline.score(X_train, y_train), 3))
print("Test  accuracy:", round(rf_pipeline.score(X_test,  y_test),  3))

## 21 — Evaluate Random Forest

In [ ]:
rf_results = evaluate("Random Forest", rf_pipeline, X_test, y_test)

## 22 — Model comparison

In [ ]:
comparison = pd.DataFrame([logreg_results, tree_results, rf_results]).round(3)
print(comparison.to_string(index=False))

comparison.set_index("Model")[["Accuracy", "Precision", "Recall", "F1"]].plot(
    kind="bar", figsize=(11, 4), rot=0, colormap="Blues_r", edgecolor="black")
plt.title("Model comparison on the test set")
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 23 — Confusion matrices side by side

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pipe) in zip(axes, [("Logistic Regression", logreg_pipeline),
                                   ("Decision Tree", tree_pipeline),
                                   ("Random Forest", rf_pipeline)]):
    ConfusionMatrixDisplay.from_estimator(
        pipe, X_test, y_test, ax=ax, cmap="Blues",
        display_labels=["Stayed", "Churned"], colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

print("Bottom-left cell = FALSE NEGATIVES = churners we failed to flag.")

## 24 — Feature importance

In [ ]:
# Pull the feature names back out of the fitted ColumnTransformer.
fitted_pre = rf_pipeline.named_steps["preprocessor"]
feature_names = fitted_pre.get_feature_names_out()

importances = pd.Series(
    rf_pipeline.named_steps["model"].feature_importances_,
    index=feature_names).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 5))
importances.sort_values().plot(kind="barh", color="#1F4E79")
plt.title("Top 15 features — Random Forest")
plt.xlabel("importance")
plt.tight_layout()
plt.show()

print(importances.round(4))

## 25 — Business interpretation: build the call list

In [ ]:
# The retention team can contact 200 customers this month.
# Lower the threshold to catch more real churners, then rank by risk.
best_pipeline = logreg_pipeline          # chosen on ROC-AUC + explainability
probabilities = best_pipeline.predict_proba(X_test)[:, 1]

risk = X_test.copy()
risk["churn_probability"] = probabilities.round(3)
risk["actual_churn"] = y_test.values
risk = risk.sort_values("churn_probability", ascending=False)

print("Top 10 highest-risk customers:")
print(risk[["tenure_months", "contract_type", "monthly_charges",
            "support_tickets", "satisfaction_score",
            "churn_probability", "actual_churn"]].head(10).to_string())

# How good is the ranking? Check the churn rate in the top 20%.
top_20pct = risk.head(int(len(risk) * 0.2))
print("\nChurn rate overall      :", round(y_test.mean(), 3))
print("Churn rate in top 20%   :", round(top_20pct["actual_churn"].mean(), 3))
print("Lift over random calling:",
      round(top_20pct["actual_churn"].mean() / y_test.mean(), 2), "x")

## 26 — Final business recommendation

In [ ]:
# Economics: a retained customer is worth Rs 18,000; an intervention costs Rs 700.
VALUE_RETAINED = 18000
COST_CONTACT = 700
RETENTION_SUCCESS_RATE = 0.30      # 30% of contacted churners are saved

print("Threshold  Flagged  TruePos  FalsePos  Recall   Expected value (Rs)")
for t in [0.20, 0.30, 0.40, 0.50, 0.60]:
    pred = (probabilities >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    value = tp * RETENTION_SUCCESS_RATE * VALUE_RETAINED - (tp + fp) * COST_CONTACT
    print(f"{t:>8.2f}  {tp+fp:>7}  {tp:>7}  {fp:>8}  "
          f"{recall_score(y_test, pred):>6.3f}   {value:>12,.0f}")

print("""
RECOMMENDATION
--------------
1. Deploy Logistic Regression. It matched or beat the tree-based models on
   ROC-AUC here, scores instantly, and every coefficient can be explained to
   a regulator or an account manager.
2. Set the threshold by expected value, not by convention. The table above
   shows which threshold maximises rupees, not accuracy.
3. Work the ranked list, not the labels. The retention team should call
   customers in descending probability order until capacity runs out.
4. Hold out a control group. Leave 10% of flagged customers uncontacted so
   the campaign's true lift can be measured. Without this you will never
   know whether the model or the market caused the change.
5. Monitor monthly: input drift, score drift, and realised churn among
   customers flagged three months ago.
""")